# Previsão de Atrasos em Entregas — Olist
Análise exploratória, pré-processamento e classificação binária com Rede Neural e Regressão Logística.

## 1. Instalação de Dependências

In [ ]:
#pip install ydata_profiling

## 2. Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ydata_profiling import ProfileReport
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

## 3. EDA — Análise Exploratória

In [ ]:
df = pd.read_csv(
    "olist_orders_dataset.csv",
    sep=",",
    on_bad_lines="skip"  # ignora linhas ruins
)

df.describe()

In [ ]:
df.info()

In [ ]:
profile = ProfileReport(df, title="Profiling Report")
profile.to_notebook_iframe()

## 4. Pré-Processamento

In [ ]:
cols_datas = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in cols_datas:
    df[col] = pd.to_datetime(df[col])

In [ ]:
# Criar target: 1 = atraso, 0 = no prazo
df['atraso'] = (df['order_delivered_customer_date'] > df['order_estimated_delivery_date']).astype(int)
df['atraso'].value_counts()

In [ ]:
df['tempo_entrega'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['tempo_estimado'] = (df['order_estimated_delivery_date'] - df['order_purchase_timestamp']).dt.days
df['diff_estimado_real'] = df['tempo_entrega'] - df['tempo_estimado']

print(df["tempo_entrega"])

In [ ]:
# Verificar nulos
contagem_entrega = df['tempo_entrega'].isna()
contagem_estimado = df['tempo_estimado'].isna()

print("Nulos em tempo_entrega:")
print(contagem_entrega.value_counts())

print("\nNulos em tempo_estimado:")
print(contagem_estimado.value_counts())

In [ ]:
# Remover nulos
df = df.dropna(subset=[
    'tempo_entrega',
    'tempo_estimado',
    'diff_estimado_real',
    'atraso'
])

## 5. Features e Normalização

In [ ]:
features = ['tempo_entrega', 'tempo_estimado', 'diff_estimado_real']

X = df[features]
y = df['atraso']

scaler = StandardScaler()
X = scaler.fit_transform(X)

## 6. Divisão Treino/Teste (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

## 7. Modelo de Rede Neural (ReLU)

In [ ]:
model = Sequential()

# Camada de entrada + oculta
model.add(Dense(16, activation='relu', input_shape=(X.shape[1],)))

# Segunda camada oculta
model.add(Dense(8, activation='relu'))

# Camada de saída (classificação binária)
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 8. Treino do Modelo

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=6,
    validation_data=(X_test, y_test)
)

## 9. Gráfico de Overfitting

In [ ]:
plt.figure()

plt.plot(history.history['loss'], label='Loss Treino')
plt.plot(history.history['val_loss'], label='Loss Validação')

if 'accuracy' in history.history:
    plt.plot(history.history['accuracy'], '--', label='Accuracy Treino')
    plt.plot(history.history['val_accuracy'], '--', label='Accuracy Validação')

plt.title('Comportamento do Modelo durante o Treinamento')
plt.xlabel('Épocas')
plt.ylabel('Métricas')
plt.legend()
plt.grid()
plt.show()

## 10. Avaliação do Modelo (ReLU)

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Acurácia:", acc)

## 11. Verificação de Overfitting

In [ ]:
train_loss = history.history['loss']
val_loss = history.history['val_loss']

diff = np.mean(np.array(val_loss) - np.array(train_loss))

print("Diferença média (val - treino):", diff)

if diff > 0.05:
    print("Indício de overfitting")
else:
    print("Sem overfitting significativo")

## 12. Modelo com Ativação Sigmoid (comparativo)

In [ ]:
model_sigmoid = Sequential()

model_sigmoid.add(Dense(16, activation='sigmoid', input_shape=(X.shape[1],)))
model_sigmoid.add(Dense(8, activation='sigmoid'))
model_sigmoid.add(Dense(1, activation='sigmoid'))

model_sigmoid.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_sigmoid = model_sigmoid.fit(
    X_train, y_train,
    epochs=7,
    validation_data=(X_test, y_test),
    verbose=0
)

loss_s, acc_s = model_sigmoid.evaluate(X_test, y_test)
print("Acurácia (Sigmoid):", acc_s)

## 13. Comparação ReLU vs Sigmoid

In [ ]:
print("ReLU final val_loss:   ", history.history['val_loss'][-1])
print("Sigmoid final val_loss:", history_sigmoid.history['val_loss'][-1])

## 14. Comparação com Regressão Logística

In [ ]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred)
print("Acurácia Regressão Logística:", acc_lr)

## 15. Conclusão

### 1. O modelo apresentou overfitting? Como você identificou isso?
Não apresentou overfitting, pois as curvas de erro de treino e validação permaneceram próximas ao longo das épocas, indicando boa generalização.

### 2. A Função ReLU contribui para o aprendizado da rede? Por quê?
Sim. A ReLU contribui ao acelerar o treinamento e evitar o problema de gradientes muito pequenos, permitindo um aprendizado mais eficiente.

### 3. Acurácia alta significa que o modelo é confiável neste contexto?
Não necessariamente. A acurácia deve ser analisada com cautela, pois pode ser influenciada por desbalanceamento ou por variáveis fortemente correlacionadas com o alvo.

### 4. Esse problema poderia ser resolvido com um modelo mais simples? Explique.
Sim. Como as variáveis possuem relação direta com o atraso, modelos mais simples podem apresentar bom desempenho, embora com menor capacidade de capturar padrões mais complexos.